<a href="https://colab.research.google.com/github/florvela/IA-y-automatizacion-en-seguridad-defensiva/blob/main/codigos-de-ejemplo/04-fundamentos-soar/04-fundamentos-soar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Módulo 04 — SOAR: Runbook vs Playbook

La diferencia fundamental es la **ejecutabilidad**.

| | Runbook | Playbook |
|---|---|---|
| Formato | Documento / Wiki | Código Python |
| Ejecutado por | Analista humano | Motor SOAR |
| Velocidad | Minutos / horas | Segundos |
| Consistencia | Variable (depende del analista) | 100% reproducible |
| Auditable | Solo si el analista documenta | Siempre (log automático) |
| Integrable con otras herramientas | No | Sí |

En este notebook vamos a ver el mismo procedimiento de respuesta a fuerza bruta SSH en ambos formatos. La plataforma que ejecuta el playbook en producción es **Shuffle** — eso lo vemos en la demo aparte.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/04-fundamentos-soar/images/cover.png" width="520"/>

In [1]:
import time
import logging
from enum import Enum
from datetime import datetime
from dataclasses import dataclass, field

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('soar-demo')
print('Setup OK')

Setup OK


## Runbook — El procedimiento como documento

Un runbook es la descripción en texto de lo que debe hacer un analista. Vive en una wiki, en Confluence, en un Google Doc. No se puede ejecutar — hay que leerlo y seguirlo a mano.

In [ ]:
RUNBOOK_FUERZA_BRUTA = """
RUNBOOK: Responder a Ataque de Fuerza Bruta SSH
================================================

PASO 1 — Recolectar información del incidente
  - Registrar la IP de origen
  - Registrar el usuario objetivo
  - Registrar cantidad de intentos y rango de tiempo

PASO 2 — Verificar si la IP es conocida
  - Buscar en la CMDB si la IP pertenece a la organización
  - Si es IP interna conocida → probable falso positivo, cerrar el caso

PASO 3 — Ejecutar mitigación
  - Bloquear la IP en el firewall perimetral
  - Forzar cambio de contraseña del usuario afectado

PASO 4 — Investigar causa raíz
  - ¿El atacante logró algún acceso exitoso?
  - Si sí → escalar a Tier 2 para análisis forense

PASO 5 — Documentar y cerrar
  - Crear ticket de incidente en Jira
  - Notificar al propietario del sistema
"""

print(RUNBOOK_FUERZA_BRUTA)

## Playbook — El mismo procedimiento, ejecutable

El playbook es el runbook convertido a código. Los **cinco pasos son los mismos**, pero ahora los ejecuta una máquina en lugar de un analista. En Shuffle, este código vive dentro de los nodos del workflow — cada nodo es un paso del runbook.

Primero definimos el modelo de datos del incidente.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/04-fundamentos-soar/images/SOAR_260321_171756.jpg" width="480"/>

In [2]:
class Severidad(str, Enum):
    CRITICA = 'CRITICAL'
    ALTA    = 'HIGH'
    MEDIA   = 'MEDIUM'

class EstadoIncidente(str, Enum):
    NUEVO          = 'nuevo'
    EN_RESPUESTA   = 'en_respuesta'
    ESCALADO       = 'escalado'
    RESUELTO       = 'resuelto'
    FALSO_POSITIVO = 'falso_positivo'

@dataclass
class Alerta:
    id: str
    ip_origen: str
    usuario: str
    intentos: int
    severidad: Severidad = Severidad.MEDIA

@dataclass
class Incidente:
    id: str
    alerta: Alerta
    estado: EstadoIncidente = EstadoIncidente.NUEVO
    acciones_tomadas: list = field(default_factory=list)
    creado_en: str = field(default_factory=lambda: datetime.now().isoformat())

    def registrar_accion(self, accion: str):
        self.acciones_tomadas.append({'accion': accion, 'ts': datetime.now().isoformat()})
        logger.info(f'[{self.id}] {accion}')

# CMDB simulada: IPs conocidas de la organización (no son amenazas)
CMDB_IPS_CONOCIDAS = {'10.0.0.1', '10.0.0.15', '192.168.1.50', '172.16.0.100'}

print('Modelo de datos definido')

Modelo de datos definido


Ahora el playbook — los mismos 5 pasos del runbook, en código:

In [3]:
def playbook_fuerza_bruta_ssh(incidente: Incidente) -> dict:
    """
    Playbook de respuesta a fuerza bruta SSH.
    Implementa exactamente los 5 pasos del runbook de arriba.
    En Shuffle, cada PASO es un nodo del workflow.
    """
    alerta = incidente.alerta
    print(f'\n{"="*55}')
    print(f' PLAYBOOK: Fuerza Bruta SSH — {incidente.id}')
    print(f'{"="*55}')

    # ── PASO 1: Recolectar información ─────────────────────
    print(f'\n[PASO 1] Recolectando información...')
    time.sleep(0.2)
    print(f'  IP origen : {alerta.ip_origen}')
    print(f'  Usuario   : {alerta.usuario}')
    print(f'  Intentos  : {alerta.intentos}')

    # ── PASO 2: Verificar CMDB ─────────────────────────────
    print(f'\n[PASO 2] Consultando CMDB...')
    time.sleep(0.2)
    es_ip_conocida = alerta.ip_origen in CMDB_IPS_CONOCIDAS
    print(f'  IP en CMDB: {es_ip_conocida}')

    if es_ip_conocida:
        incidente.estado = EstadoIncidente.FALSO_POSITIVO
        incidente.registrar_accion('Cerrado como falso positivo — IP interna conocida')
        print('  → Falso positivo. Cerrando caso.')
        return {'resultado': 'falso_positivo', 'acciones': incidente.acciones_tomadas}

    # ── PASO 3: Mitigar ────────────────────────────────────
    print(f'\n[PASO 3] Ejecutando mitigación...')
    time.sleep(0.3)
    incidente.estado = EstadoIncidente.EN_RESPUESTA
    incidente.registrar_accion(f'IP {alerta.ip_origen} bloqueada en firewall')
    incidente.registrar_accion(f'Password reset forzado para usuario {alerta.usuario}')
    print(f'  ✓ IP {alerta.ip_origen} bloqueada')
    print(f'  ✓ Password reset iniciado para {alerta.usuario}')

    # ── PASO 4: Investigar causa raíz ──────────────────────
    print(f'\n[PASO 4] Buscando accesos exitosos desde esta IP...')
    time.sleep(0.2)
    # En producción: query al SIEM buscando 'Accepted' desde esta IP
    acceso_exitoso = alerta.intentos > 100
    if acceso_exitoso:
        incidente.estado = EstadoIncidente.ESCALADO
        incidente.registrar_accion('Escalado a Tier 2: posible acceso exitoso')
        print('  ⚠️  Posible acceso exitoso — escalando a Tier 2')
    else:
        print('  ✓ Sin accesos exitosos detectados')

    # ── PASO 5: Documentar y cerrar ────────────────────────
    print(f'\n[PASO 5] Creando ticket...')
    time.sleep(0.1)
    ticket_id = f'JIRA-{incidente.id}'
    incidente.registrar_accion(f'Ticket creado: {ticket_id}')
    print(f'  ✓ Ticket: {ticket_id}')

    if incidente.estado != EstadoIncidente.ESCALADO:
        incidente.estado = EstadoIncidente.RESUELTO

    print(f'\n→ Estado final : {incidente.estado}')
    print(f'  Acciones      : {len(incidente.acciones_tomadas)}')
    return {'resultado': incidente.estado, 'ticket': ticket_id,
            'acciones': incidente.acciones_tomadas}


# ── Ejecutar con datos de ejemplo ──────────────────────────
alerta = Alerta(
    id='ALT-2024-001',
    ip_origen='173.234.31.186',
    usuario='root',
    intentos=47,
    severidad=Severidad.ALTA
)
incidente = Incidente(id='INC-2024-001', alerta=alerta)
resultado = playbook_fuerza_bruta_ssh(incidente)


 PLAYBOOK: Fuerza Bruta SSH — INC-2024-001

[PASO 1] Recolectando información...
  IP origen : 173.234.31.186
  Usuario   : root
  Intentos  : 47

[PASO 2] Consultando CMDB...
  IP en CMDB: False

[PASO 3] Ejecutando mitigación...
  ✓ IP 173.234.31.186 bloqueada
  ✓ Password reset iniciado para root

[PASO 4] Buscando accesos exitosos desde esta IP...
  ✓ Sin accesos exitosos detectados

[PASO 5] Creando ticket...
  ✓ Ticket: JIRA-INC-2024-001

→ Estado final : EstadoIncidente.RESUELTO
  Acciones      : 3
